In [1]:
from PIL import Image, ImageSequence

In [2]:
def image_to_bitmap(path, width=16, height=16, invert=False):
    # Cargar imagen
    img = Image.open(path)

    # Convertir a blanco y negro
    img = img.convert("1")  # 1-bit (blanco/negro)

    # Redimensionar
    img = img.resize((width, height))

    pixels = img.load()

    bitmap = []

    for y in range(height):
        row_bits = ""

        for x in range(width):
            pixel = pixels[x, y]

            # En modo "1", 0 = negro, 255 = blanco
            bit = 0 if pixel else 1

            if invert:
                bit = 1 - bit

            row_bits += str(bit)

        # Dividir en bloques de 8 bits
        byte1 = row_bits[:8]
        byte2 = row_bits[8:]

        bitmap.append(f"0b{byte1}, 0b{byte2},")

    return bitmap


def print_bitmap(bitmap):
    print("static const unsigned char PROGMEM bitmap[] = {")
    for line in bitmap:
        print("  " + line)
    print("};")


In [3]:
bitmap = image_to_bitmap("creeper.jpg", 16, 16)
print_bitmap(bitmap)

static const unsigned char PROGMEM bitmap[] = {
  0b00000000, 0b00000000,
  0b00000000, 0b00000000,
  0b00111100, 0b00111100,
  0b00111100, 0b00111100,
  0b00111100, 0b00111100,
  0b00111100, 0b00111100,
  0b00000011, 0b11000000,
  0b00000011, 0b11000000,
  0b00001111, 0b11110000,
  0b00001111, 0b11110000,
  0b00001111, 0b11110000,
  0b00001111, 0b11110000,
  0b00001100, 0b00110000,
  0b00001100, 0b00110000,
  0b00000000, 0b00000000,
  0b00000000, 0b00000000,
};


In [4]:
def image_to_bitmap_2(path, width=16, height=16, invert=False):
    # Cargar imagen
    img = Image.open(path)

    # Convertir a blanco y negro
    img = img.convert("1")  # 1-bit (blanco/negro)

    # Redimensionar
    img = img.resize((width, height))

    pixels = img.load()

    bitmap = []

    for y in range(height):
        row_bits = ""

        for x in range(width):
            pixel = pixels[x, y]

            # En modo "1", 0 = negro, 255 = blanco
            bit = 0 if pixel else 1

            if invert:
                bit = 1 - bit

            row_bits += str(bit)

        # Dividir en bloques de 8 bits
        #byte1 = row_bits[:8]
        #byte2 = row_bits[8:]

        bitmap.append(f"{row_bits}")

    return bitmap


def print_bitmap(bitmap):
    print("static const unsigned char PROGMEM bitmap[] = {")
    for line in bitmap:
        print("  " + line)
    print("};")


In [5]:
bitmap = image_to_bitmap_2("creeper.jpg", 8, 8)
print_bitmap(bitmap)

static const unsigned char PROGMEM bitmap[] = {
  00000000
  01100110
  01100110
  00011000
  00111100
  00111100
  00100100
  00000000
};


In [6]:
def frame_to_bitmap(img, width=16, height=16, invert=False):
    img = img.convert("1")  # blanco y negro
    img = img.resize((width, height))

    pixels = img.load()
    bitmap = []

    for y in range(height):
        row_bits = ""
        for x in range(width):
            pixel = pixels[x, y]

            bit = 0 if pixel else 1  # negro = 1
            if invert:
                bit = 1 - bit

            row_bits += str(bit)

        byte1 = row_bits[:8]
        byte2 = row_bits[8:]
        bitmap.append(f"0b{byte1}, 0b{byte2},")

    return bitmap


In [7]:
def gif_to_header(gif_path, output_path, name="anim", width=16, height=16, invert=False):
    im = Image.open(gif_path)

    frames = []
    for frame in ImageSequence.Iterator(im):
        bitmap = frame_to_bitmap(frame, width, height, invert)
        frames.append(bitmap)

    with open(output_path, "w") as f:
        f.write("#ifndef ANIM_BITMAP_H\n")
        f.write("#define ANIM_BITMAP_H\n\n")
        f.write("#include <avr/pgmspace.h>\n\n")

        f.write(f"#define {name.upper()}_WIDTH {width}\n")
        f.write(f"#define {name.upper()}_HEIGHT {height}\n")
        f.write(f"#define {name.upper()}_FRAMES {len(frames)}\n\n")

        for i, frame in enumerate(frames):
            f.write(f"const unsigned char {name}_frame_{i}[] PROGMEM = {{\n")
            for line in frame:
                f.write(f"  {line}\n")
            f.write("};\n\n")

        # Array de punteros
        f.write(f"const unsigned char* {name}_frames[] = {{\n")
        for i in range(len(frames)):
            f.write(f"  {name}_frame_{i},\n")
        f.write("};\n\n")

        f.write("#endif\n")

    print(f"Header generado: {output_path}")




In [9]:

gif_to_header(
    gif_path="Corazon_animacion.gif",
    output_path="animacion.h",
    name="mi_animacion",
    width=16,
    height=16,
    invert=False
)

Header generado: animacion.h


In [10]:
from PIL import Image, ImageSequence

def frame_to_hex_matrix(img, size=8, invert=False):
    img = img.convert("1")  # blanco y negro
    img = img.resize((size, size))

    pixels = img.load()

    frame_data = []

    for y in range(size):
        byte = 0

        for x in range(size):
            pixel = pixels[x, y]

            # Negro = LED encendido
            bit = 0 if pixel else 1

            if invert:
                bit = 1 - bit

            # Construir byte
            byte |= (bit << (7 - x))

        frame_data.append(f"0x{byte:02X}")

    return frame_data


def gif_to_arduino_matrix(
        gif_path,
        output_path,
        array_name="anim",
        size=8,
        invert=False):

    im = Image.open(gif_path)

    frames = []

    for frame in ImageSequence.Iterator(im):
        frame_data = frame_to_hex_matrix(frame, size, invert)
        frames.append(frame_data)

    with open(output_path, "w") as f:

        f.write("#include <Arduino.h>\n\n")

        f.write(
            f"const uint8_t {array_name}"
            f"[{len(frames)}][{size}] PROGMEM = {{\n"
        )

        for i, frame in enumerate(frames):

            frame_string = ", ".join(frame)

            if i < len(frames) - 1:
                f.write(f"  {{{frame_string}}},\n")
            else:
                f.write(f"  {{{frame_string}}}\n")

        f.write("};\n")

    print(f"Archivo generado: {output_path}")
    print(f"Frames: {len(frames)}")

In [11]:
gif_to_arduino_matrix(
    gif_path="Corazon_animacion.gif",
    output_path="animacion.h",
    array_name="miAnimacion",
    size=8
)

Archivo generado: animacion.h
Frames: 24
